# 06-2. 誤差逆伝播法 — 動かして確かめる

📖 解説: [`../02_backprop.md`](../02_backprop.md)

## このノートで触るもの
1. 連鎖律の手書き計算 (1層)
2. NumPy で手書き backprop (2層 MLP)
3. JAX で同じことを 1 行で
4. 勾配の値を手書き vs JAX で比較
5. 完全な MLP 訓練 (Optax 使用)

> 🧭 **クイックナビ**: 📚 [ROOT (全体 TOP)](../../README.md) ・ 🏠 [章 TOP](../README.md) ・ 📖 [解説 md (02_backprop.md)](../02_backprop.md)

In [ ]:
import numpy as np
import sympy as sp
import jax
import jax.numpy as jnp
import optax
import matplotlib.pyplot as plt

%matplotlib inline
rng = np.random.default_rng(42)

## 1. 連鎖律の手書き計算 (線形回帰)

$$
\hat{y} = w x + b, \quad L = (y - \hat{y})^2
$$

$$
\frac{\partial L}{\partial w} = -2x(y - wx - b), \quad \frac{\partial L}{\partial b} = -2(y - wx - b)
$$

In [ ]:
# SymPy で確認
w, b, x, y = sp.symbols('w b x y')
L = (y - (w * x + b))**2
print(f'∂L/∂w = {sp.simplify(sp.diff(L, w))}')
print(f'∂L/∂b = {sp.simplify(sp.diff(L, b))}')

## 2. NumPy で手書き backprop (2層 MLP)

$$
\mathbf{z}_1 = W_1 \mathbf{x} + \mathbf{b}_1, \quad \mathbf{h}_1 = \text{ReLU}(\mathbf{z}_1)
$$

$$
\hat{\mathbf{y}} = W_2 \mathbf{h}_1 + \mathbf{b}_2, \quad L = \frac{1}{2}\|\hat{\mathbf{y}} - \mathbf{y}\|^2
$$

In [ ]:
def relu(z):
    return np.maximum(0, z)

def relu_grad(z):
    return (z > 0).astype(float)

def forward(x, W1, b1, W2, b2):
    """順伝播."""
    z1 = W1 @ x + b1
    h1 = relu(z1)
    y_hat = W2 @ h1 + b2
    return z1, h1, y_hat

def backward_manual(x, y, z1, h1, y_hat, W1, W2):
    """手書き backprop."""
    # 損失から出力
    dL_dy_hat = y_hat - y
    
    # 出力層パラメータ
    dL_dW2 = np.outer(dL_dy_hat, h1)
    dL_db2 = dL_dy_hat
    
    # 隠れ層へ
    dL_dh1 = W2.T @ dL_dy_hat
    dL_dz1 = dL_dh1 * relu_grad(z1)
    
    # 入力層パラメータ
    dL_dW1 = np.outer(dL_dz1, x)
    dL_db1 = dL_dz1
    
    return dL_dW1, dL_db1, dL_dW2, dL_db2

# 実行例
d_in, d_hidden, d_out = 4, 3, 2
W1 = rng.normal(0, 0.5, (d_hidden, d_in))
b1 = rng.normal(0, 0.5, d_hidden)
W2 = rng.normal(0, 0.5, (d_out, d_hidden))
b2 = rng.normal(0, 0.5, d_out)

x = rng.normal(0, 1, d_in)
y = rng.normal(0, 1, d_out)

z1, h1, y_hat = forward(x, W1, b1, W2, b2)
loss_val = 0.5 * np.sum((y_hat - y)**2)
print(f'損失: {loss_val:.6f}')

grads_manual = backward_manual(x, y, z1, h1, y_hat, W1, W2)
print(f'\n手書き勾配 shapes: W1={grads_manual[0].shape}, b1={grads_manual[1].shape}, W2={grads_manual[2].shape}, b2={grads_manual[3].shape}')

## 3. JAX で 1 行に — backprop が消える

対応するコード: `jax.grad(loss)(params, x, y)` だけ

In [ ]:
def model_jax(params, x):
    W1, b1, W2, b2 = params
    h1 = jax.nn.relu(W1 @ x + b1)
    return W2 @ h1 + b2

def loss_jax(params, x, y):
    y_hat = model_jax(params, x)
    return 0.5 * jnp.sum((y_hat - y)**2)

params_jax = (jnp.array(W1), jnp.array(b1), jnp.array(W2), jnp.array(b2))
x_jax = jnp.array(x)
y_jax = jnp.array(y)

grads_jax = jax.grad(loss_jax)(params_jax, x_jax, y_jax)
print('JAX 勾配 shapes:', [g.shape for g in grads_jax])

## 4. 手書き vs JAX の検算

勾配の値が一致するか確認。理論上 1e-6 以下なら OK。

In [ ]:
for name, manual, auto in zip(
    ['W1', 'b1', 'W2', 'b2'],
    grads_manual,
    grads_jax,
):
    diff = np.max(np.abs(manual - np.array(auto)))
    status = '✅' if diff < 1e-6 else '❌'
    print(f'{status} {name}: 最大誤差 {diff:.2e}')

→ 完全に一致! `jax.grad` は手書き backprop を**完全に再現**している。

違いは:
- 手書き: 1 つの NN ごとに数十行のコードが必要
- JAX: `jax.grad(loss)` だけで OK

## 5. 完全な MLP 訓練 (Optax + Adam)

実用パターン: 損失定義 → `jax.grad` で勾配 → Optax で更新

In [ ]:
# データ: y = sin(x) 回帰
n = 200
X_data = jnp.array(rng.uniform(-3, 3, n))
y_data = jnp.sin(X_data) + 0.1 * jnp.array(rng.normal(0, 1, n))

# モデル: 2層 MLP
def init_params(key, d_in=1, d_hidden=20, d_out=1):
    keys = jax.random.split(key, 4)
    return {
        'W1': jax.random.normal(keys[0], (d_hidden, d_in)) * 0.5,
        'b1': jnp.zeros(d_hidden),
        'W2': jax.random.normal(keys[2], (d_out, d_hidden)) * 0.5,
        'b2': jnp.zeros(d_out),
    }

def model(params, x):
    h = jax.nn.tanh(params['W1'] @ jnp.array([x]) + params['b1'])
    return (params['W2'] @ h + params['b2'])[0]

def loss(params, x, y):
    pred = jax.vmap(lambda xi: model(params, xi))(x)
    return jnp.mean((pred - y)**2)

# 訓練
key = jax.random.PRNGKey(0)
params = init_params(key)
optimizer = optax.adam(1e-2)
opt_state = optimizer.init(params)

@jax.jit
def step(params, opt_state, x, y):
    val, grads = jax.value_and_grad(loss)(params, x, y)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    return params, opt_state, val

history = []
for epoch in range(2000):
    params, opt_state, val = step(params, opt_state, X_data, y_data)
    history.append(float(val))
    if epoch % 400 == 0:
        print(f'epoch {epoch:4d}  loss = {float(val):.6f}')

# 可視化
x_grid = jnp.linspace(-3, 3, 100)
y_pred = jax.vmap(lambda xi: model(params, xi))(x_grid)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history)
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('損失')
axes[0].set_yscale('log')
axes[0].set_title('損失の推移 (Adam)')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(X_data, y_data, alpha=0.4, s=15, label='データ')
axes[1].plot(x_grid, jnp.sin(x_grid), 'g--', alpha=0.7, label='真の関数 sin(x)')
axes[1].plot(x_grid, y_pred, 'r-', linewidth=2, label='MLP 予測')
axes[1].set_title('NN による sin(x) の学習')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## まとめ
- 連鎖律で勾配を「逆向きに」伝播するのが backprop
- 手書きは数十行 → JAX なら `jax.grad(loss)` 1 行
- 手書きと JAX の勾配は完全一致 (内部で同じことをやっている)
- 完全な NN 訓練 = JAX + Optax で 20 行程度

## ML 橋渡し章 完了 🎉

→ 次の章: [`../../07_jax/README.md`](../../07_jax/README.md)

あるいは離散数学に進む: [`../../04_discrete_math/README.md`](../../04_discrete_math/README.md)

---

## 📍 ナビゲーション

| ← 前 | 🏠 章 TOP | 📚 全体 TOP | 次の章 → |
|---|---|---|---|
| [`01_loss_functions.ipynb`](01_loss_functions.ipynb) | [章 TOP](../README.md) | [📚 ROOT README](../../README.md) | [`../../07_jax/README.md`](../../07_jax/README.md) |